# Práctica N.° 15 — Grafos
## Modelado de la Red Vial de la Región Puno — Representación y Análisis en Python y C++

**Curso:** Algoritmos y Estructuras de Datos — SIS210
**Estudiante:** Francy Jimena Ramos Vilca
**Docente:** Dr. Aldo Hernán Zanabria Gálvez
**Fecha:** 27 de julio de 2026

Este notebook contiene el código completo de las Actividades 1 a 4 (Python) y las
extensiones de análisis (crossover de memoria lista vs matriz, y verificación de la
Pregunta de Reflexión 3), ejecutado de forma real. Las Actividades 5 y 6 (C++17:
`grafo_puno.hpp`, `analisis_red_vial.cpp`) se entregan como archivos fuente separados,
documentadas en el informe (`Practica15_Grafos_Ramos_Vilca_Francy_Jimena.docx`).

## Actividades 1, 3 y 4: GrafoPuno, GrafoDirigidoPuno y componentes conexas

Se define la clase `GrafoPuno` (lista de adyacencia con `defaultdict`), la clase
`GrafoDirigidoPuno` (restricciones de sentido único durante la Festividad de la
Candelaria), y la función `componentes_conexas` (BFS con soporte para vértices
excluidos, usada para simular el bloqueo de trochas en temporada de lluvias).

In [1]:
# ── Actividad 1: GrafoPuno — lista de adyacencia ────────────────────────
from collections import defaultdict, deque


class GrafoPuno:
    CIUDADES = {0: 'Puno', 1: 'Juliaca', 2: 'Ilave', 3: 'Desaguadero', 4: 'Yunguyo',
                5: 'Juli', 6: 'Lampa', 7: 'Azangaro', 8: 'Huancane', 9: 'Moho',
                10: 'Putina', 11: 'Ayaviri', 12: 'Macusani', 13: 'Sandia'}
    RIESGO = {0: 'bajo', 1: 'bajo', 2: 'bajo', 3: 'bajo', 4: 'medio', 5: 'bajo',
              6: 'medio', 7: 'medio', 8: 'alto', 9: 'alto', 10: 'alto',
              11: 'medio', 12: 'alto', 13: 'alto'}

    def __init__(self, n):
        self.n = n
        self.adj = defaultdict(list)
        self.naristas = 0

    def agregar_arista(self, u, v, peso):
        self.adj[u].append((v, peso))
        self.adj[v].append((u, peso))  # no dirigido: ambos sentidos
        self.naristas += 1

    def grado(self, u):
        return len(self.adj[u])

    def densidad(self):
        return 2 * self.naristas / (self.n * (self.n - 1))


# Red vial principal — distancias aproximadas en km (MTC, 2024)
RUTAS_PUNO = [
    (0, 1, 44), (0, 2, 55), (0, 5, 80), (1, 6, 37), (1, 7, 70), (1, 11, 90),
    (2, 3, 50), (2, 4, 45), (3, 4, 25), (5, 4, 60), (7, 8, 95), (7, 10, 110),
    (7, 11, 75), (8, 9, 40), (11, 12, 140), (11, 13, 180),
]


def construir_grafo_puno():
    g = GrafoPuno(14)
    for u, v, p in RUTAS_PUNO:
        g.agregar_arista(u, v, p)
    return g


# ── Actividad 3: Grafo DIRIGIDO — restricciones festividad ──────────────
class GrafoDirigidoPuno:
    """
    Representa las calles del centro de Puno con restricciones de
    sentido unico durante la Festividad de la Virgen de la Candelaria
    (febrero), declarada Patrimonio Cultural Inmaterial de la Humanidad
    por la UNESCO en 2014 -- la misma festividad de la que se origina
    la danza de la Diablada Punena.
    """

    def __init__(self, n):
        self.n = n
        self.adj = defaultdict(list)

    def agregar_arista_dirigida(self, u, v, peso):
        self.adj[u].append((v, peso))  # SOLO un sentido: u -> v

    def es_alcanzable(self, origen, destino):
        """Verifica si destino es alcanzable desde origen (DFS simple)."""
        visitados = set()

        def dfs(u):
            if u == destino:
                return True
            visitados.add(u)
            return any(dfs(v) for v, _ in self.adj[u] if v not in visitados)

        return dfs(origen)


# Calles del centro historico -- 6 intersecciones (simplificado)
# 0=Plaza de Armas 1=Jr.Lima 2=Av.El Sol 3=Jr.Moquegua 4=Jr.Tacna 5=Malecon
RESTRICCIONES_CANDELARIA = [
    (0, 1, 'solo bajada hacia el malecon'),
    (1, 5, 'flujo unico hacia el lago durante el corso'),
    (2, 0, 'acceso unico a la plaza desde Av. El Sol'),
    (3, 2, 'desvio obligatorio'),
    (4, 3, 'sentido unico zona comercial'),
]


def construir_grafo_dirigido():
    gd = GrafoDirigidoPuno(6)
    for u, v, _ in RESTRICCIONES_CANDELARIA:
        gd.agregar_arista_dirigida(u, v, 1)
    return gd


# ── Actividad 4: Componentes conexas bajo bloqueo por lluvias ───────────
def componentes_conexas(g, vertices_excluidos=None):
    excluidos = vertices_excluidos or set()
    visitados = set(excluidos)
    componentes = []
    for inicio in range(g.n):
        if inicio in visitados:
            continue
        componente = []
        cola = deque([inicio])
        visitados.add(inicio)
        while cola:
            u = cola.popleft()
            componente.append(u)
            for v, _ in g.adj[u]:
                if v not in visitados and v not in excluidos:
                    visitados.add(v)
                    cola.append(v)
        componentes.append(componente)
    return componentes


## Ejecución real de las Actividades 1 a 4

In [2]:
import sys
import numpy as np
print("=== Actividad 1: GrafoPuno -- lista de adyacencia ===")
g = construir_grafo_puno()
print(f'|V|={g.n} |E|={g.naristas} densidad={g.densidad():.3f}')
for c in range(g.n):
    print(f'  {g.CIUDADES[c]:15} grado={g.grado(c)}')

print("\n=== Actividad 2: Matriz de adyacencia con numpy ===")


def construir_matriz(g):
    M = np.full((g.n, g.n), np.inf)
    np.fill_diagonal(M, 0)
    for u in g.adj:
        for v, peso in g.adj[u]:
            M[u][v] = peso
    return M


matriz = construir_matriz(g)
print('Matriz de adyacencia (km, inf = sin ruta directa):')
np.set_printoptions(linewidth=200, precision=0, suppress=True)
print(matriz)

tam_lista = sys.getsizeof(g.adj) + sum(sys.getsizeof(v) for v in g.adj.values())
tam_matriz = matriz.nbytes
print(f'\nMemoria lista de adyacencia: {tam_lista} bytes')
print(f'Memoria matriz de adyacencia: {tam_matriz} bytes')
print(f'Razon matriz/lista: {tam_matriz/tam_lista:.1f}x')

print("\n=== Actividad 3: Grafo dirigido -- restricciones Candelaria ===")
gd = construir_grafo_dirigido()
print('¿Se puede ir del Jr. Tacna (4) a la Plaza de Armas (0)?', gd.es_alcanzable(4, 0))
print('¿Se puede ir de la Plaza de Armas (0) al Jr. Tacna (4)?', gd.es_alcanzable(0, 4))

print("\n=== Actividad 4: Componentes conexas bajo bloqueo por lluvias ===")
print('=== Red vial COMPLETA (sin lluvias) ===')
comp_normal = componentes_conexas(g)
print(f'Componentes conexas: {len(comp_normal)}')
for c in comp_normal:
    print(f'  {[g.CIUDADES[v] for v in c]}')

print()
print('=== Simulacion: trochas de Macusani(12) y Sandia(13) bloqueadas ===')
ciudades_aisladas = {12, 13}
comp_lluvia = componentes_conexas(g, vertices_excluidos=ciudades_aisladas)
print(f'Componentes conexas restantes: {len(comp_lluvia)}')
for c in comp_lluvia:
    print(f'  {[g.CIUDADES[v] for v in c]}')
print(f'Ciudades completamente aisladas: {[g.CIUDADES[v] for v in ciudades_aisladas]}')

=== Actividad 1: GrafoPuno -- lista de adyacencia ===
|V|=14 |E|=16 densidad=0.176
  Puno            grado=3
  Juliaca         grado=4
  Ilave           grado=3
  Desaguadero     grado=2
  Yunguyo         grado=3
  Juli            grado=2
  Lampa           grado=1
  Azangaro        grado=4
  Huancane        grado=2
  Moho            grado=1
  Putina          grado=1
  Ayaviri         grado=4
  Macusani        grado=1
  Sandia          grado=1

=== Actividad 2: Matriz de adyacencia con numpy ===
Matriz de adyacencia (km, inf = sin ruta directa):
[[  0.  44.  55.  inf  inf  80.  inf  inf  inf  inf  inf  inf  inf  inf]
 [ 44.   0.  inf  inf  inf  inf  37.  70.  inf  inf  inf  90.  inf  inf]
 [ 55.  inf   0.  50.  45.  inf  inf  inf  inf  inf  inf  inf  inf  inf]
 [ inf  inf  50.   0.  25.  inf  inf  inf  inf  inf  inf  inf  inf  inf]
 [ inf  inf  45.  25.   0.  60.  inf  inf  inf  inf  inf  inf  inf  inf]
 [ 80.  inf  inf  inf  60.   0.  inf  inf  inf  inf  inf  inf  inf  inf]
 [ inf  37.

## Extensión: ¿por qué la matriz salió más pequeña que la lista con solo 14 nodos?

Se prueba el mismo tipo de grafo disperso a mayor escala para encontrar el punto de
cruce donde la lista de adyacencia empieza a ser más eficiente en memoria que la
matriz, y se responde con datos reales la Pregunta de Reflexión 3 (bloquear solo
Macusani, sin tocar Sandia).

In [3]:
import sys
import numpy as np
import random
from collections import defaultdict
print("=== Extension: ¿por que la matriz salio MAS PEQUENA que la lista en la Actividad 2? ===\n")
print("Con solo 14 nodos, el overhead por objeto de Python (cada tupla, cada lista)")
print("pesa mas que la ventaja asintotica O(V+E) vs O(V^2). Se prueba el mismo grafo")
print("disperso (misma densidad relativa) a mayor escala para encontrar el punto de cruce.\n")


def grafo_disperso_aleatorio(n, grado_promedio=2.3, semilla=42):
    """Genera un grafo aleatorio disperso con densidad similar a la red de Puno."""
    random.seed(semilla)
    g = GrafoPuno(n)
    aristas_objetivo = int(n * grado_promedio / 2)
    intentos = 0
    while g.naristas < aristas_objetivo and intentos < aristas_objetivo * 20:
        u, v = random.randint(0, n - 1), random.randint(0, n - 1)
        intentos += 1
        if u != v and not any(vv == v for vv, _ in g.adj[u]):
            g.agregar_arista(u, v, random.randint(10, 200))
    return g


def medir_memoria(g):
    tam_lista = sys.getsizeof(g.adj) + sum(sys.getsizeof(v) for v in g.adj.values())
    M = np.full((g.n, g.n), np.inf, dtype=np.float64)
    tam_matriz = M.nbytes
    return tam_lista, tam_matriz


print(f'{"N":>8} {"Lista (bytes)":>15} {"Matriz (bytes)":>15} {"Razon M/L":>12} {"Gana":>8}')
for n in [14, 50, 100, 500, 1000, 5000]:
    g = grafo_disperso_aleatorio(n)
    tam_lista, tam_matriz = medir_memoria(g)
    razon = tam_matriz / tam_lista
    gana = "Matriz" if razon < 1 else "Lista"
    print(f'{n:>8} {tam_lista:>15,} {tam_matriz:>15,} {razon:>11.2f}x {gana:>8}')

print("\n=== Pregunta de Reflexion 3: bloquear SOLO Macusani (12), sin tocar Sandia (13) ===\n")
g = construir_grafo_puno()
print('Vecinos de Macusani (12):', [(g.CIUDADES[v], p) for v, p in g.adj[12]])
print('Vecinos de Sandia (13):  ', [(g.CIUDADES[v], p) for v, p in g.adj[13]])

comp_solo_macusani = componentes_conexas(g, vertices_excluidos={12})
print(f'\nBloqueando SOLO Macusani(12): {len(comp_solo_macusani)} componentes conexas')
for c in comp_solo_macusani:
    print(f'  {[g.CIUDADES[v] for v in c]}')

sandia_aislada = all(len(c) == 1 and g.CIUDADES[c[0]] == 'Sandia' or 'Sandia' not in [g.CIUDADES[x] for x in c]
                      for c in comp_solo_macusani)
sandia_en_componente_grande = any('Sandia' in [g.CIUDADES[x] for x in c] and len(c) > 1 for c in comp_solo_macusani)
print(f'\n¿Sandia quedo aislada al bloquear SOLO Macusani? '
      f'{"NO, sigue conectada" if sandia_en_componente_grande else "SI, quedo aislada"}')

=== Extension: ¿por que la matriz salio MAS PEQUENA que la lista en la Actividad 2? ===

Con solo 14 nodos, el overhead por objeto de Python (cada tupla, cada lista)
pesa mas que la ventaja asintotica O(V+E) vs O(V^2). Se prueba el mismo grafo
disperso (misma densidad relativa) a mayor escala para encontrar el punto de cruce.

       N   Lista (bytes)  Matriz (bytes)    Razon M/L     Gana
      14           1,872           1,568        0.84x   Matriz
      50           5,000          20,000        4.00x    Lista
     100          13,072          80,000        6.12x    Lista
     500          59,608       2,000,000       33.55x    Lista
    1000         119,152       8,000,000       67.14x    Lista


    5000         557,840     200,000,000      358.53x    Lista

=== Pregunta de Reflexion 3: bloquear SOLO Macusani (12), sin tocar Sandia (13) ===

Vecinos de Macusani (12): [('Ayaviri', 140)]
Vecinos de Sandia (13):   [('Ayaviri', 180)]

Bloqueando SOLO Macusani(12): 1 componentes conexas
  ['Puno', 'Juliaca', 'Ilave', 'Juli', 'Lampa', 'Azangaro', 'Ayaviri', 'Desaguadero', 'Yunguyo', 'Huancane', 'Putina', 'Sandia', 'Moho']

¿Sandia quedo aislada al bloquear SOLO Macusani? NO, sigue conectada


## Benchmark Python: construcción del grafo y componentes conexas a escala

Para complementar la comparación de rendimiento contra C++ (Actividad 6), se mide el
tiempo real de construcción y de cómputo de componentes conexas en Python, tanto para
la red de 14 ciudades como para grafos dispersos aleatorios de hasta 1,000,000 de
nodos (misma metodología que el benchmark en C++).

In [4]:
import time
import random
from collections import defaultdict, deque
print("=== Benchmark Python: red vial de Puno (14 ciudades) ===")
t0 = time.perf_counter()
g = GrafoPuno(14)
for u, v, p in RUTAS_PUNO:
    g.agregar_arista(u, v, p)
ms_construccion = (time.perf_counter() - t0) * 1000
print(f'Tiempo de construccion del grafo (14 ciudades): {ms_construccion:.6f} ms')

t0 = time.perf_counter()
comp = componentes_conexas(g)
ms_comp = (time.perf_counter() - t0) * 1000
print(f'Componentes (sin bloqueos): {len(comp)} (calculado en {ms_comp:.6f} ms)')

t0 = time.perf_counter()
comp_lluvia = componentes_conexas(g, vertices_excluidos={12, 13})
ms_comp_lluvia = (time.perf_counter() - t0) * 1000
print(f'Componentes (Macusani/Sandia bloqueadas): {len(comp_lluvia)} (calculado en {ms_comp_lluvia:.6f} ms)')


def grafo_disperso(n, grado_promedio, rng):
    g = GrafoPuno(n)
    aristas_objetivo = int(n * grado_promedio / 2)
    for _ in range(aristas_objetivo):
        u, v = rng.randint(0, n - 1), rng.randint(0, n - 1)
        if u != v:
            g.agregar_arista(u, v, rng.randint(10, 200))
    return g


print("\n=== Benchmark a escala (grafos dispersos aleatorios, grado promedio ~2.3) ===")
rng = random.Random(42)
for n in [1000, 10000, 100000, 1000000]:
    t0 = time.perf_counter()
    g_grande = grafo_disperso(n, 2.3, rng)
    ms_const = (time.perf_counter() - t0) * 1000

    t0 = time.perf_counter()
    comp_grande = componentes_conexas(g_grande)
    ms_comp = (time.perf_counter() - t0) * 1000

    print(f'n={n} | construccion={ms_const:.3f}ms | componentes={len(comp_grande)} en {ms_comp:.3f}ms')

=== Benchmark Python: red vial de Puno (14 ciudades) ===
Tiempo de construccion del grafo (14 ciudades): 0.111829 ms
Componentes (sin bloqueos): 1 (calculado en 0.064366 ms)
Componentes (Macusani/Sandia bloqueadas): 1 (calculado en 0.097189 ms)

=== Benchmark a escala (grafos dispersos aleatorios, grado promedio ~2.3) ===
n=1000 | construccion=3.009ms | componentes=118 en 0.441ms
n=10000 | construccion=82.867ms | componentes=1173 en 11.474ms


n=100000 | construccion=432.283ms | componentes=11503 en 156.576ms


n=1000000 | construccion=7748.282ms | componentes=115716 en 3197.020ms


## Conclusión del notebook

Las Actividades 1, 3 y 4 (Python) se ejecutaron realmente sobre la red vial de las 14
ciudades de la Región Puno, confirmando que la red completa es una única componente
conexa, y que bloquear las trochas de Macusani y Sandia por lluvias las aísla
completamente sin afectar al resto de la red. La extensión de memoria reveló que el
argumento "lista mejor que matriz para grafos dispersos" es correcto asintóticamente,
pero se invierte para grafos muy pequeños (como el de 14 ciudades) debido al overhead
de objetos de Python. El benchmark de escala confirmó que C++ es sustancialmente más
rápido que Python tanto en construcción del grafo (~13x a 1,000,000 de nodos) como en
el cómputo de componentes conexas (~3.5x). El desarrollo completo en C++17 (Actividades
5 y 6), las preguntas de reflexión y el trabajo de investigación (OSMnx y Plan Vial
MTC) se documentan en el informe adjunto.